# 08 — KV cache and efficient generation (M2.2)

How a decoder-only model generates text one token at a time, why recomputing the
whole prefix is wasteful, and how a **KV cache** removes that waste without changing
the result. Everything runs on the random **Tiny Qwen3** in seconds (CPU). The
optional last section repeats the check on the real Qwen3-0.6B-Base
(`QWEN3_WEIGHTS_DIR=... QWEN3_NB_PRETRAINED=1`, ~4 GiB RAM).

Write-up: `docs/components/kv_cache.md`. Benchmark script: `scripts/benchmark_generation.py`.

In [ ]:
import dataclasses
import os
import time

import torch

from qwen3_scratch.config import QWEN3_0_6B_BASE, QWEN3_TINY
from qwen3_scratch.generation import (
    GenerationConfig,
    apply_top_k,
    apply_top_p,
    generate,
    select_next_token,
)
from qwen3_scratch.kv_cache import kv_cache_bytes
from qwen3_scratch.layers.attention import causal_mask
from qwen3_scratch.model import Qwen3ForCausalLM

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

cfg = dataclasses.replace(QWEN3_TINY, max_position_embeddings=512)
_ = torch.manual_seed(0)
model = Qwen3ForCausalLM(
    cfg, dtype=torch.float64
).eval()  # float64: differences = bugs, not rounding
prompt = torch.randint(0, cfg.vocab_size, (1, 6), generator=torch.Generator().manual_seed(1))
print(cfg)

## 1. Autoregressive generation

A causal LM gives, at every position $t$, a distribution over the **next** token:
$p(x_{t+1} \mid x_0..x_t)$. Generation repeats: run the model, pick a token from the
last position's distribution, append it, run again.

## 2. Full-prefix recomputation (what M1.6 did)

Without a cache, step $t$ feeds the whole sequence of length $T_0 + t$ through every
layer. The projections and the MLP cost $O(T)$ per step and attention $O(T^2)$. Yet
only the **last** position's output is new: every earlier position produces exactly
the same keys, values and outputs as in the previous step, because of causality.

In [ ]:
seq = prompt
with torch.inference_mode():
    for step in range(4):
        logits = model(seq).logits  # recomputes ALL positions
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        print(
            f"step {step}: processed {seq.shape[1]:2d} tokens to produce 1 new token -> {int(nxt)}"
        )
        seq = torch.cat([seq, nxt], dim=1)

## 3. Prefill vs decode, and why K/V can be reused

In layer $\ell$, the key and value of token $j$ are
$k_j = \mathrm{RoPE}_j(\mathrm{RMSNorm}(W_k h_j))$, $v_j = W_v h_j$, where $h_j$ is the
layer input at position $j$. With a causal mask, $h_j$ depends only on tokens $0..j$, so
**$k_j, v_j$ never change once computed**. Store them:

* **prefill**: run the prompt $[B, T_0]$ once, store K/V of all $T_0$ positions per layer;
* **decode**: feed only the new token $[B, 1]$; it computes its own $q, k, v$, appends
  $k, v$ to the cache and attends over all $T_0 + t$ cached keys.

**Why not cache queries?** A query is only used to compute *its own* position's output.
Old positions' outputs were already produced (and their contribution to later layers is
captured by later layers' cached K/V), so old queries are never needed again.

In [ ]:
cache = model.new_cache()
with torch.inference_mode():
    out = model(prompt, past_key_values=cache, use_cache=True)  # prefill
    print("after prefill :", cache)
    nxt = out.logits[:, -1].argmax(-1, keepdim=True)
    for step in range(3):
        out = model(nxt, past_key_values=cache, use_cache=True)  # decode: ONE token in
        shapes = f"input {tuple(nxt.shape)}, logits {tuple(out.logits.shape)}"
        print(f"decode step {step}: {shapes}, cache length {cache.seq_len}")
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
k0, v0 = cache.get(0)
print("layer-0 keys", tuple(k0.shape), "= [B, Nkv, T_cached, D]")

## 4. Cache tensor shapes and memory (GQA)

Per layer we store K and V as `[B, Nkv, T, D]`, with the **native** number of KV heads.
Grouped-query attention expands them to `Nq` heads only inside the attention computation
(`repeat_kv`), never in the cache.

$$\text{KV bytes} = 2 \cdot L \cdot B \cdot N_{kv} \cdot T \cdot D \cdot \text{bytes per element}$$

Qwen3-0.6B: $L=28$, $N_{kv}=8$, $D=128$, giving 112 KiB per token in bf16. With 16 KV heads
(plain multi-head attention, $N_{kv}=N_q$) it would be twice as much.

In [ ]:
for t in (128, 2048, 32768):
    gqa = kv_cache_bytes(QWEN3_0_6B_BASE, t, bytes_per_element=2)
    mha = kv_cache_bytes(
        dataclasses.replace(QWEN3_0_6B_BASE, num_key_value_heads=16), t, bytes_per_element=2
    )
    print(
        f"T={t:6d}: GQA (8 KV heads) {gqa / 2**20:8.1f} MiB | MHA (16 heads) {mha / 2**20:8.1f} MiB"
    )
if plt is not None:
    ts = [2**k for k in range(6, 16)]
    fig, ax = plt.subplots(figsize=(6, 3))
    for b, ls in ((1, "-"), (8, "--")):
        ax.plot(
            ts,
            [kv_cache_bytes(QWEN3_0_6B_BASE, t, batch_size=b) / 2**30 for t in ts],
            ls,
            color="#2a78d6",
            label=f"batch {b}",
        )
    ax.axhline(1.11, color="#999", lw=1)
    ax.text(ts[0], 1.2, "bf16 weights (1.11 GiB)", fontsize=8, color="#555")
    ax.set(
        xscale="log",
        yscale="log",
        xlabel="context length T",
        ylabel="KV cache (GiB, bf16)",
        title="Qwen3-0.6B KV cache grows linearly with T and B",
    )
    ax.legend()
    plt.show()

## 5. RoPE with position offsets

The new token at index 6 must be rotated with **position 6**, not 0. The model takes
the offset from `cache.seq_len`. Cached keys are stored **after** RoPE, so they are never
rotated again. Getting either wrong changes the result:

In [ ]:
ids = torch.cat([prompt, torch.tensor([[17]])], dim=1)  # 7 tokens; the last one is "new"
with torch.inference_mode():
    full = model(ids).logits[:, -1]
    c = model.new_cache()
    model(ids[:, :6], past_key_values=c, use_cache=True)
    right = model(ids[:, 6:], past_key_values=c, use_cache=True).logits[:, -1]
    c = model.new_cache()
    model(ids[:, :6], past_key_values=c, use_cache=True)
    wrong = model(
        ids[:, 6:], position_ids=torch.tensor([[0]]), past_key_values=c, use_cache=True
    ).logits[:, -1]
print(f"correct position 6 : max |diff| vs full forward = {(right - full).abs().max():.2e}")
print(f"wrong position 0   : max |diff| vs full forward = {(wrong - full).abs().max():.2e}")

## 6. Cached attention masking: $T_q \ne T_k$

Visibility is defined by **absolute** indices: query $i$ may see key $j$ iff $j \le i$.
With a cache, queries are the new tokens at indices $T_{past}..T_{past}+T-1$ and keys are
all $T_{past}+T$ tokens, so the mask is a $T \times (T_{past}+T)$ matrix:

* prefill ($T_{past}=0$): the usual lower triangle;
* single-token decode ($T=1$): one row, **all ones**, since the newest token sees everything incl. itself;
* chunked decode ($T>1$): the past is fully visible, and the chunk is causal inside.

A plain `tril(ones(T, T_k))` would be wrong here: it aligns the triangle to the *top-left*
instead of the *bottom-right*, hiding most of the past from the new queries.

In [ ]:
def show(t_past, t_new):
    q = torch.arange(t_past, t_past + t_new)
    k = torch.arange(t_past + t_new)
    return causal_mask(q, k).int()


cases = {
    "prefill (0 past, 5 new)": show(0, 5),
    "decode (5 past, 1 new)": show(5, 1),
    "chunk (4 past, 3 new)": show(4, 3),
}
for name, m in cases.items():
    print(name)
    print(m)
naive = torch.tril(torch.ones(3, 7)).int()
print("WRONG: torch.tril(ones(3, 7)) for the chunk case\n", naive)
if plt is not None:
    fig, axes = plt.subplots(1, 3, figsize=(9, 2.6))
    for ax, (name, m) in zip(axes, cases.items(), strict=True):
        ax.imshow(m.numpy(), cmap="Blues", vmin=0, vmax=1.4)
        ax.set(title=name, xlabel="key index", ylabel="query index")
        ax.set_xticks(range(m.shape[1]))
        ax.set_yticks(range(m.shape[0]))
    fig.tight_layout()
    plt.show()

## 7. Generation algorithms

From the last logits $z$ (float32), `select_next_token` applies, in this order:

1. **greedy** (`do_sample=False` or `temperature=0`): $\arg\max_i z_i$;
2. **temperature** $\tau$: $z / \tau$; $\tau<1$ sharpens, $\tau>1$ flattens;
3. **top-k**: keep the $k$ largest logits;
4. **top-p (nucleus)**: keep the smallest set of most-likely tokens with total
   probability $\ge p$ (the top-1 token is always kept);
5. softmax (max-subtracted, so stable) and `torch.multinomial` with a seeded generator.

In [ ]:
z = torch.tensor([[3.0, 2.5, 2.0, 1.0, 0.5, 0.0, -1.0, -2.0]])
p = lambda x: torch.softmax(x, -1)[0]  # noqa: E731
rows = {
    "tau=1": p(z),
    "tau=0.5": p(z / 0.5),
    "tau=2": p(z / 2),
    "top-k=3": p(apply_top_k(z, 3)),
    "top-p=0.8": p(apply_top_p(z, 0.8)),
}
for name, r in rows.items():
    print(f"{name:9s}", " ".join(f"{v:.3f}" for v in r.tolist()))
g = torch.Generator().manual_seed(0)
cfgs = GenerationConfig(do_sample=True, temperature=0.8, top_k=5, top_p=0.9)
print("samples:", [int(select_next_token(z, cfgs, g)) for _ in range(12)])
if plt is not None:
    fig, axes = plt.subplots(1, len(rows), figsize=(12, 2.2), sharey=True)
    for ax, (name, r) in zip(axes, rows.items(), strict=True):
        ax.bar(range(len(r)), r.tolist(), color="#2a78d6")
        ax.set_title(name, fontsize=9)
        ax.set_xticks(range(len(r)))
    axes[0].set_ylabel("probability")
    fig.tight_layout()
    plt.show()

## 8. Cached vs uncached: numerical comparison

The cache must not change the model, only the work. The mandatory check: logits of one
full forward pass equal the logits of prefill + token-by-token cached decoding.
In float64, the only differences come from summation order, around 1e-15.

In [ ]:
ids = torch.randint(0, cfg.vocab_size, (2, 20), generator=torch.Generator().manual_seed(5))
with torch.inference_mode():
    full = model(ids).logits
    cache = model.new_cache()
    parts = [model(ids[:, :8], past_key_values=cache, use_cache=True).logits]
    parts += [
        model(ids[:, t : t + 1], past_key_values=cache, use_cache=True).logits for t in range(8, 20)
    ]
cached = torch.cat(parts, 1)
print(f"float64: max |full - cached| = {(full - cached).abs().max():.2e}")

a = generate(model, prompt, max_new_tokens=16, use_cache=True, return_step_logits=True)
b = generate(model, prompt, max_new_tokens=16, use_cache=False, return_step_logits=True)
print("greedy ids identical:", torch.equal(a.sequences, b.sequences))
print(
    "max per-step logit diff:",
    max(float((x - y).abs().max()) for x, y in zip(a.step_logits, b.step_logits, strict=True)),
)

## 9. Latency (tiny model, this machine)

A small, bounded benchmark of the generation loop. Caching makes each decode step
process **one** token instead of the whole sequence. It does **not** make attention
free: the new query still attends over all $T$ cached keys, so the per-step cost still
grows (slowly) with context length. Use `scripts/benchmark_generation.py` for real numbers.

In [ ]:
model32 = Qwen3ForCausalLM(cfg).eval()


def timed(use_cache, prompt_len, n=24, repeats=3):
    p = torch.randint(
        0, cfg.vocab_size, (1, prompt_len), generator=torch.Generator().manual_seed(0)
    )
    runs = []
    for _ in range(repeats):  # median of a few runs: single timings on a laptop are noisy
        t = time.perf_counter()
        generate(model32, p, max_new_tokens=n, use_cache=use_cache)
        runs.append((time.perf_counter() - t) / n * 1e3)
    return sorted(runs)[len(runs) // 2]


timed(True, 8, n=4), timed(False, 8, n=4)  # warm-up (first calls pay one-off setup costs)
lens = [16, 64, 128, 256]
res = {m: [timed(m == "cached", L) for L in lens] for m in ("cached", "uncached")}
for m, v in res.items():
    print(f"{m:8s} ms/token:", " ".join(f"{x:6.2f}" for x in v), f"(prompt lengths {lens})")
if plt is not None:
    fig, ax = plt.subplots(figsize=(5, 3))
    for m, c in (("cached", "#2a78d6"), ("uncached", "#d9822b")):
        ax.plot(lens, res[m], "o-", color=c, label=m)
    ax.set(
        xlabel="prompt length",
        ylabel="ms per generated token (incl. prefill)",
        title="Tiny Qwen3, CPU",
    )
    ax.legend()
    plt.show()

## 10. Common KV cache bugs (and the test that catches each)

| bug | symptom | test |
|---|---|---|
| new token gets position 0 | wrong logits from the 2nd generated token on | `test_cached_decode_uses_nonzero_positions` |
| cached keys rotated again each step | errors grow with step count | `test_attention_stores_native_kv_heads_post_rope` |
| `tril(T, T_k)` mask when $T_q \ne T_k$ | new queries cannot see the past | `test_chunked_decoding_matches_full` |
| storing repeated (Nq) heads | correct, but 2x memory for Qwen3-0.6B | `..._native_kv_heads_...` |
| appending the prompt twice | cache length too long, positions shifted | `test_cache_does_not_reprocess_old_tokens` |
| reusing a cache across sessions | the second prompt "sees" the first | `test_reset_and_reuse_gives_fresh_session` |
| half-updated cache after an error | later steps misaligned | `test_failed_forward_rolls_back` |
| padding with a cache | silently wrong positions | rejected with `NotImplementedError` |

## 11. Optional: the real Qwen3-0.6B-Base

In [ ]:
if os.environ.get("QWEN3_NB_PRETRAINED") == "1" and os.environ.get("QWEN3_WEIGHTS_DIR"):
    from qwen3_scratch.pretrained import load_pretrained
    from qwen3_scratch.tokenizer import Qwen3Tokenizer

    real, _ = load_pretrained(os.environ["QWEN3_WEIGHTS_DIR"], dtype=torch.float32)
    tok = Qwen3Tokenizer.from_pretrained(os.environ["QWEN3_WEIGHTS_DIR"])
    ids = torch.tensor([tok.encode("Artificial intelligence is")])
    a = generate(real, ids, max_new_tokens=16, use_cache=True, return_step_logits=True)
    b = generate(real, ids, max_new_tokens=16, use_cache=False, return_step_logits=True)
    print(repr(tok.decode(a.new_tokens[0])))
    print(
        "ids equal:",
        torch.equal(a.sequences, b.sequences),
        "| max step logit diff:",
        max(float((x - y).abs().max()) for x, y in zip(a.step_logits, b.step_logits, strict=True)),
    )
else:
    print("skipped (set QWEN3_WEIGHTS_DIR and QWEN3_NB_PRETRAINED=1 to run on the real model)")

## Exercises

1. Why does the cached decode step still get slower as the context grows?
2. How large is the KV cache of Qwen3-0.6B for batch 4 and 8,192 tokens in bf16?
3. Implement a pre-allocated cache (fixed `[B, Nkv, T_max, D]` buffers and a write index)
   and check it against `KVCache` with the full-vs-cached test.